In [0]:
from pyspark.sql import functions as F

df_silver = spark.sql("SELECT * FROM files.amazon_files.amazon_silver")

In [0]:
df_com_faixa = df_silver.withColumn(
    "faixa_desconto",
    F.when(F.col("discount_percentage") < 0.25, "0-25%")
    .when(F.col("discount_percentage") < 0.50, "25-50%")
    .when(F.col("discount_percentage") < 0.75, "50-75%")
    .otherwise("75-100%") 
)

In [0]:
df_com_faixa.select("discount_percentage", "faixa_desconto").show(10)

+-------------------+--------------+
|discount_percentage|faixa_desconto|
+-------------------+--------------+
|               0.64|        50-75%|
|               0.43|        25-50%|
|                0.9|       75-100%|
|               0.53|        50-75%|
|               0.61|        50-75%|
|               0.85|       75-100%|
|               0.65|        50-75%|
|               0.23|         0-25%|
|                0.5|        50-75%|
|               0.33|        25-50%|
+-------------------+--------------+
only showing top 10 rows


In [0]:
df_gold_desconto = df_com_faixa.groupBy("faixa_desconto") \
    .agg(
        F.count("*").alias("qtd_produtos"),
        F.expr("percentile_approx(rating, 0.5)").alias("mediana_rating")
    ) \
    .orderBy(F.desc("mediana_rating"))

df_gold_desconto.show()

+--------------+------------+--------------+
|faixa_desconto|qtd_produtos|mediana_rating|
+--------------+------------+--------------+
|         0-25%|         222|           4.2|
|        25-50%|         449|           4.2|
|        50-75%|         527|           4.1|
|       75-100%|         155|           4.0|
+--------------+------------+--------------+



In [0]:
df_gold_desconto.write.format("delta").mode("overwrite").saveAsTable("files.amazon_files.amazon_gold_desconto")

In [0]:
spark.sql("SELECT * FROM files.amazon_files.amazon_gold_desconto").show()

+--------------+------------+--------------+
|faixa_desconto|qtd_produtos|mediana_rating|
+--------------+------------+--------------+
|         0-25%|         222|           4.2|
|        25-50%|         449|           4.2|
|        50-75%|         527|           4.1|
|       75-100%|         155|           4.0|
+--------------+------------+--------------+

